# Свечи по лучшим trial (несколько методов / пациентов)

Запуск по порядку: **настройка PYTHONPATH** → **загрузка `.db` и `best_df`** → **график**.

Правило имён: `tfr_<patient>_<method>.db` (суффикс метода может содержать `_`, напр. `tfr_svm`).

Детальный разбор одного study см. `open_study_and_visualise.ipynb`.


In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_cands = [_cwd, _cwd.parent, *_cwd.parents[:3]]
project_root = next((p for p in _cands if (p / "lib" / "optuna").is_dir()), None)
if project_root is None:
    raise FileNotFoundError(
        "NeuronDeCo root not found (expected directory with lib/optuna). "
        f"cwd={_cwd}"
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from IPython.display import display
from optuna.trial import TrialState

from lib.optuna import load_study_sqlite

print("project_root:", project_root)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from optuna.trial import TrialState

from lib.optuna import load_study_sqlite


def select_best_trial_top5_f1_then_min_loss(study):
    """Top-5 by F1, then choose one with minimum loss."""
    complete_trials = [
        t
        for t in study.get_trials(deepcopy=False)
        if t.state == TrialState.COMPLETE and t.values is not None and len(t.values) >= 2
    ]
    if not complete_trials:
        return None

    ranked_by_f1 = sorted(complete_trials, key=lambda t: float(t.values[0]), reverse=True)
    top5 = ranked_by_f1[:5]
    best = min(top5, key=lambda t: float(t.values[1]))
    return best


# Несколько каталогов: один .stem — последний перечисленный каталог побеждает.
# method = суффикс после tfr_<patient>_ (transformer, alexnet, tfr_svm, …).
RUN_DIRS = (
    Path("../../PreprocessedData/2026-04-01"),
    # Path("../../PreprocessedData/2026-05-10"),
)

by_stem: dict[str, Path] = {}
for d in RUN_DIRS:
    d = Path(d).expanduser().resolve()
    if not d.is_dir():
        continue
    for p in sorted(d.glob("tfr_*_*.db")):
        by_stem[p.stem] = p.resolve()

db_paths = sorted(by_stem.values(), key=lambda p: p.stem)
if not db_paths:
    raise FileNotFoundError(f"No studies found under RUN_DIRS={RUN_DIRS!r}")

study_rows = []
best_rows = []

for db_path in db_paths:
    stem = db_path.stem
    parts = stem.split("_")
    if len(parts) < 3:
        continue
    patient_id = parts[1]
    method = "_".join(parts[2:])

    study = load_study_sqlite(db_path=db_path, study_name=stem)

    complete_trials = [
        t
        for t in study.get_trials(deepcopy=False)
        if t.state == TrialState.COMPLETE and t.values is not None and len(t.values) >= 2
    ]
    for t in complete_trials:
        study_rows.append(
            {
                "patient": patient_id,
                "method": method,
                "trial_number": int(t.number),
                "f1": float(t.values[0]),
                "loss": float(t.values[1]),
            }
        )

    best_trial = select_best_trial_top5_f1_then_min_loss(study)
    if best_trial is not None:
        best_rows.append(
            {
                "patient": patient_id,
                "method": method,
                "trial_number": int(best_trial.number),
                "f1": float(best_trial.values[0]),
                "loss": float(best_trial.values[1]),
                "params": best_trial.params,
                "db_path": str(db_path),
            }
        )

trials_df = pd.DataFrame(study_rows)
best_df = pd.DataFrame(best_rows).sort_values(["patient", "method"]).reset_index(drop=True)

print(f"Loaded studies: {len(db_paths)}")
print(f"Complete trial rows: {len(trials_df)}")
print("\nBest trial per patient/method (top5 F1 -> min loss):")
display(best_df[["patient", "method", "trial_number", "f1", "loss", "params"]])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import cm as mpl_cm
from matplotlib.lines import Line2D
from pathlib import Path

from lib.optuna import load_study_sqlite


if best_df.empty:
    raise RuntimeError("No best trials. Run previous cell first.")

# Столбцы на графике (порядок). None → все методы из best_df + хвост «не упомянутые».
METHODS_ORDER: list[str] | None = ["alexnet", "transformer", "tfr_svm"]


def _last_epoch_f1_per_fold(row) -> np.ndarray:
    """
    F1 на последней эпохе каждого фолда (одно число на фолд).
    Holdout → одна точка на свечу; CV → несколько.
    """
    db_path = Path(row["db_path"])
    study = load_study_sqlite(db_path=db_path, study_name=db_path.stem)
    trial_num = int(row["trial_number"])
    tr = next(t for t in study.get_trials(deepcopy=False) if int(t.number) == trial_num)
    fold_curves = tr.user_attrs.get("fold_curves", None) or []
    vals: list[float] = []
    for fc in fold_curves:
        f1s = fc.get("val_f1s", []) or []
        if f1s:
            vals.append(float(f1s[-1]))
    if vals:
        return np.asarray(vals, dtype=np.float64)
    flat = tr.user_attrs.get("val_f1s", []) or []
    if flat:
        return np.asarray([float(flat[-1])], dtype=np.float64)
    return np.asarray([], dtype=np.float64)


_default_method_labels = {
    "alexnet": "AlexNet",
    "transformer": "Transformer",
    "tfr_svm": "TFR + SVM",
}
_default_method_colors = {
    "alexnet": (0.0, 0.447, 0.741),
    "transformer": (0.85, 0.325, 0.098),
    "tfr_svm": (0.466, 0.674, 0.188),
}


def _label_for_method(m: str) -> str:
    if m in _default_method_labels:
        return _default_method_labels[m]
    return " ".join(p.capitalize() for p in m.replace("_", " ").split())


def _color_for_method(m: str) -> tuple[float, float, float]:
    if m in _default_method_colors:
        return _default_method_colors[m]
    tab = mpl_cm.get_cmap("tab10")
    hue = abs(hash(m)) % 5237 / 5237.0
    return tuple(float(x) for x in tab(hue)[:3])

lookup = best_df.set_index(["patient", "method"])
patient_order = sorted(best_df["patient"].unique())
uniq_methods = sorted(best_df["method"].unique())
if METHODS_ORDER is None:
    methods_order = uniq_methods
else:
    head = [m for m in METHODS_ORDER if m in set(uniq_methods)]
    tail = [m for m in uniq_methods if m not in head]
    methods_order = head + tail

if not methods_order:
    raise RuntimeError("methods_order empty; check METHODS_ORDER and best_df")

n_m = len(methods_order)
fig_w_scale = max(1.0, 0.45 * (n_m - 1))
fig, ax = plt.subplots(figsize=(max(12.0, len(patient_order) * 1.35 * fig_w_scale), 7.0))

ax.set_axisbelow(True)
ax.grid(which="major", axis="both", linestyle="-", linewidth=0.6, alpha=0.25)
ax.grid(which="minor", axis="both", linestyle="-", linewidth=0.35, alpha=0.12)
ax.minorticks_on()

base_x = np.arange(len(patient_order), dtype=float)
if n_m <= 1:
    shifts = np.array([0.0])
else:
    step = float(min(0.22, 0.82 / max(n_m - 1, 1)))
    shifts = (np.arange(n_m, dtype=float) - (n_m - 1) / 2.0) * step
box_width = float(min(0.22, 0.06 + 0.16 / np.sqrt(max(n_m, 2))))

all_y = []

for m_idx, method in enumerate(methods_order):
    color = _color_for_method(method)
    shift = float(shifts[m_idx])

    for p_idx, patient in enumerate(patient_order):
        key = (patient, method)
        if key not in lookup.index:
            continue
        row = lookup.loc[key]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]

        f1_vals = _last_epoch_f1_per_fold(row)
        if f1_vals.size == 0:
            continue

        x = base_x[p_idx] + shift
        vmin = float(np.min(f1_vals))
        vmax = float(np.max(f1_vals))
        q1 = float(np.quantile(f1_vals, 0.25))
        q3 = float(np.quantile(f1_vals, 0.75))
        med = float(np.median(f1_vals))
        all_y.extend([vmin, vmax])

        ax.plot([x, x], [vmin, vmax], color=color, linewidth=1.2, alpha=0.95, zorder=2)
        ax.scatter([x, x], [vmin, vmax], color=color, s=22, zorder=3)
        ax.plot([x, x], [q1, q3], color=color, linewidth=7.0, alpha=0.88, zorder=4)
        ax.plot([x - box_width / 2, x + box_width / 2], [med, med], color="black", linewidth=2.2, zorder=5)

ax.set_xticks(base_x)
ax.set_xticklabels(patient_order)
ax.set_xlabel("Patient")
ax.set_ylabel("F1")
ax.set_title(
    "Best trial (top-5 F1 → min loss): last-epoch val F1 per fold\n"
    "(holdout = one fold → one value; whisker=min..max, body=Q1..Q3, black=median)"
)

if all_y:
    ymin, ymax = min(all_y), max(all_y)
    pad = max(0.02, 0.08 * (ymax - ymin if ymax > ymin else 1.0))
    ax.set_ylim(ymin - pad, ymax + pad)

legend_handles = [
    Line2D([0], [0], color=_color_for_method(m), lw=5, label=_label_for_method(m))
    for m in methods_order
    if not best_df[(best_df["method"] == m)].empty
]
ax.legend(handles=legend_handles, loc="best", frameon=True)

plt.tight_layout()
plt.show()